# Fine-tuning

In [ ]:
import os
import json
import pandas as pd
import time
from tqdm.notebook import tqdm
from IPython.display import clear_output
import matplotlib.pyplot as plt
from mistralai import Mistral

# Configuration
ds_id = "ds1"  # dataset id of training data
train_file_path = f"data/ft_{ds_id}_train.jsonl"
val_file_path = "data/ft_val.jsonl"
test_file_path = "data/ft_test.jsonl"
test_results_path = f"results/ft_test_{ds_id}_results.jsonl"

ft_run_name = f"{ds_id}_3b"
wandb_project_name = "cpc"

lr = 0.00001
n_epochs = 1
model_to_ft = "ministral-3b-latest"



### Start fine-tuning job

In [ ]:
client = Mistral(api_key=os.environ.get("MISTRAL"))

# Upload training data
training_data = client.files.upload(
    file={
        "file_name": train_file_path.split("/")[-1],
        "content": open(train_file_path, "rb"),
    }
)
# Upload validation data
validation_data = client.files.upload(
    file={
        "file_name": val_file_path.split("/")[-1],
        "content": open(val_file_path, "rb"),
    }
)
print("Uploaded training and validation data")

In [ ]:
# Create a fine-tuning job
created_job = client.fine_tuning.jobs.create(
    model=model_to_ft,
    job_type="classifier",
    training_files=[{"file_id": training_data.id, "weight": 1}],
    validation_files=[validation_data.id],
    suffix=ds_id,
    hyperparameters={"epochs": n_epochs, "learning_rate": lr},
    auto_start=False,
    integrations=[
        {
            "project": wandb_project_name,
            "api_key": os.environ.get("WANDB_API_KEY"),
            "run_name": ft_run_name,
        }
    ],
)
print(f"Created fine-tuning job: {created_job.id}")

In [ ]:
# Retrieve the job details
retrieved_job = client.fine_tuning.jobs.get(job_id=created_job.id)
print(json.dumps(retrieved_job.model_dump(), indent=4))

# Wait for the job to be validated
while retrieved_job.status not in ["VALIDATED"]:
    retrieved_job = client.fine_tuning.jobs.get(job_id=created_job.id)

    clear_output(wait=True)  # Clear the previous output
    print("Waiting for job to be validated...")
    print(json.dumps(retrieved_job.model_dump(), indent=4))
    time.sleep(1)
clear_output()
print(
    f"Job validated. Expected duration in seconds: {retrieved_job.metadata.expected_duration_seconds}. Cost in {retrieved_job.metadata.cost_currency}: {retrieved_job.metadata.cost}"
)

In [ ]:
# Start the fine-tuning job
client.fine_tuning.jobs.start(job_id=created_job.id)

# Retrieve the job status again
retrieved_job = client.fine_tuning.jobs.get(job_id=created_job.id)
print(f"Job status: {retrieved_job.status}")

### Plot loss curve

In [ ]:
# Plot loss curve
# Initialize DataFrames to store the metrics
train_metrics_df = pd.DataFrame(columns=["Step Number", "Train Loss"])
valid_metrics_df = pd.DataFrame(columns=["Step Number", "Valid Loss"])

# Total training steps
total_training_steps = retrieved_job.hyperparameters.training_steps

# Wait for the job to complete
while retrieved_job.status in ["QUEUED", "RUNNING"]:
    retrieved_job = client.fine_tuning.jobs.get(job_id=created_job.id)

    if retrieved_job.status == "QUEUED":
        time.sleep(5)
        continue

    # Clear the previous output
    clear_output(wait=True)
    print(retrieved_job.status)

    # Extract metrics from all checkpoints
    for checkpoint in retrieved_job.checkpoints[::-1]:
        metrics = checkpoint.metrics
        step_number = checkpoint.step_number

        # Check if the step number is already in the DataFrame
        if step_number not in train_metrics_df["Step Number"]:
            # Prepare the new row for train loss
            train_row = {
                "Step Number": step_number,
                "Train Loss": metrics.train_loss,
            }

            # Append the new train metrics to the DataFrame
            train_metrics_df = pd.concat(
                [train_metrics_df, pd.DataFrame([train_row])], ignore_index=True
            )

            # Prepare the new row for valid loss if available
            if metrics.valid_loss != 0:
                valid_row = {
                    "Step Number": step_number,
                    "Valid Loss": metrics.valid_loss,
                }
                # Append the new valid metrics to the DataFrame
                valid_metrics_df = pd.concat(
                    [valid_metrics_df, pd.DataFrame([valid_row])], ignore_index=True
                )

    if len(retrieved_job.checkpoints) > 0:
        # Sort the DataFrames by step number
        train_metrics_df = train_metrics_df.sort_values(by="Step Number")
        valid_metrics_df = valid_metrics_df.sort_values(by="Step Number")

        # Plot the evolution of train loss and valid loss
        plt.figure(figsize=(10, 6))

        # Plot train loss
        plt.plot(
            train_metrics_df["Step Number"],
            train_metrics_df["Train Loss"],
            label="Train Loss",
            linestyle="-",
        )

        # Highlight start and end points of train loss
        plt.scatter(
            train_metrics_df.iloc[[0, -1]]["Step Number"],
            train_metrics_df.iloc[[0, -1]]["Train Loss"],
            color="blue",
            zorder=5,
        )

        # Plot valid loss only if available
        if not valid_metrics_df.empty:
            plt.plot(
                valid_metrics_df["Step Number"],
                valid_metrics_df["Valid Loss"],
                label="Valid Loss",
                linestyle="--",
            )

            # Highlight start and end points of valid loss
            plt.scatter(
                valid_metrics_df.iloc[[0, -1]]["Step Number"],
                valid_metrics_df.iloc[[0, -1]]["Valid Loss"],
                color="orange",
                zorder=5,
            )

        plt.xlabel("Step Number")
        plt.ylabel("Loss")
        plt.title("Train Loss and Valid Loss")
        plt.legend()
        plt.grid(True)
        plt.show()

    time.sleep(1)

### Inference on test set

In [ ]:
# Load the test samples
with open(test_file_path, "r") as f:
    test_samples = [json.loads(line) for line in f.readlines()]


def classify_sample(sample_text):
    classifier_response = client.classifiers.classify(
        model=retrieved_job.fine_tuned_model,
        inputs=[sample_text],
    )
    return classifier_response


# Classify the first test sample
classifier_response = classify_sample(test_samples[0]["text"])

print("Predicted class IDs:", json.dumps(classifier_response.model_dump(), indent=4))
print("Patent description:", test_samples[0]["text"])

In [ ]:
# Classify all test samples
test_results = []
for i, sample in enumerate(tqdm(test_samples)):
    predicted_class_ids = classify_sample(sample["text"])
    test_results.append({"custom_id": i, "pred_class_ids": predicted_class_ids})
    print("Patent description:", sample["text"])
    print("Predicted class IDs:", predicted_class_ids)
    print("\n")

# Store in results jsonl file
with open(test_results_path, "w") as f:
    for result in test_results:
        f.write(json.dumps(result) + "\n")
print(f'To evaluate, run `eval.py -f "ft" -d {ds_id} -r {test_results_path}`')